In [ ]:
import warnings
warnings.filterwarnings("ignore")
import sys
import os
import torch
sys.path.append(rf'/home/{os.getlogin()}/data_share')
import FactorFramework.FactorFramework as ff

In [ ]:
# 1. 参考get_base_data_fea.py计算fea格式的基础字段数据。计算完成后，data_share/Stock60sBaseDataAll/Feather下应该有以改字段系列名称命名的子目录，子目录中包含【日期.fea】格式的每日基础字段文件

In [ ]:
# 2. 使用ff.fea_to_mmap将fea格式的文件转换为内存映射文件
# 第一个参数是基础字段系列名，conv_start_date和conv_end_date是转换的起止日期
ff.fea_to_mmap_user('Test1', conv_start_date='20210104', conv_end_date='20240930')

In [ ]:
# 3. 转换完成后，可以确认一下新的字段是否已经注册
ff.load_all_base_data() # 重新读取最新的基础字段文件
ff.base_data_config.Base_data_dict.keys() # 显示所有基础字段

In [ ]:
# 1. 定义算子函数
# 框架中涉及到基础字段和其他算子的返回值类型均为torch.Tensor
# 入参和返回的Tensor的0轴是时间，1轴是股票，返回值应当与入参Tensor形状相同
# 算子的入参根据算子的功能可能是Tensor，int，float，str等，但一定会有至少一个Tensor
# 时序算子请以ts_前缀命名，截面算子请以cs_前缀命名，其他逐元素算子（例如加减乘除）不需要时序或截面前缀

In [ ]:
# 时序算子
# 以ts_corr_sample算子为例，这个算子计算x和y在过去d分钟的相关系数，是每个股票在自身时序上的相关系数
# 时序算子最开始的d-1分钟因为数据量不足，结果应当为空值，d分钟开始有非空值
# 不需要在时序算子的前面d-1分钟填补空值。实际计算时，框架会使用当天截止当前分钟的可得数据或昨日数据来避免空值问题，在算子中填补前d-1分钟的空值会造成干扰
def ts_corr_sample(x: torch.Tensor, y: torch.Tensor, d: int) -> torch.Tensor:
    res = torch.full(x.shape, torch.nan, dtype=x.dtype, device=x.device)
    x_unfold = x.unfold(0, d, 1)
    y_unfold = y.unfold(0, d, 1)
    x_demean = x_unfold - x_unfold.nanmean(dim=-1, keepdim=True)
    y_demean = y_unfold - y_unfold.nanmean(dim=-1, keepdim=True)
    x_std = torch.sqrt(torch.nansum(torch.pow(x_demean, 2), dim=-1))
    y_std = torch.sqrt(torch.nansum(torch.pow(y_demean, 2), dim=-1))
    numerator = (x_demean * y_demean).nansum(dim=-1)
    denominator = x_std * y_std
    res[d - 1:] = numerator / denominator
    res[d - 1:][(x_std < 1e-9) | (y_std < 1e-9)] = torch.nan
    return res